# UdaciMed | Notebook 3: Hardware Acceleration & Production Deployment

Welcome to the final phase of UdaciMed's optimization pipeline! In this notebook, you will implement cross-platform hardware acceleration techniques and strategize for the deployment of your optimized model across hardware targets.

## Recap: Optimization Journey

In [Notebook 2](02_architecture_optimization.ipynb), you have implemented architectural optimizations that brought you closer to your optimization targets.

Now, it is time to unlock further performance opportunities with hardware acceleration.

> **Your mission**: Transform your optimized model into a production-ready cross-platform deployment that meets production SLAs on this reference hardware, and finalize UdaciMed's deployment strategy across its diverse hardware fleet.

### Hardware acceleration

You will implement and evaluate **2 core deployment techniques\*** using [ONNX Runtime](https://onnxruntime.ai/):

1. **Mixed Precision (FP16)** - Utilizing 16-bit floating-point numbers to significantly speed up calculations and reduce memory usage on compatible hardware.
2. **Dynamic Batching** - Finding the best batch size to maximize throughput for offline tasks while maintaining low latency for real-time requests.

Additionally, you will analyze three deployment scenarios: GPU (TensorRT), CPU (OpenVINO), and Edge deployment considerations.

_\* Note that while you are expected to implement both deployment techniques, you can decide whether to keep either or both in your final deployment strategy to best achieve targets._

---

Through this notebook, you will:

- **Convert PyTorch model to ONNX** for cross-platform deployment
- **Apply hardware acceleration using ONNX Runtime** on the reference T4 device
- **Benchmark end-to-end performance** against SLAs
- **Validate clinical safety** across the deployment pipeline
- **Analyze alternative deployment strategies** for diverse hardware environments

**Let's deliver a production-ready, hardware-accelerated diagnostic deployment!**

## Step 1: Setup the environment

First, let's set up the environment and understand our reference hardware capabilities. 

This ensures our optimization and benchmarking code will run smoothly.

In [1]:
# Make sure that libraries are dynamically re-loaded if changed
%load_ext autoreload
%autoreload 2

In [2]:
# Import core libraries
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
import pickle
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Literal
import warnings
warnings.filterwarnings('ignore')

# Import project utilities
from utils.data_loader import (
    load_pneumoniamnist,
    get_sample_batch
)
from utils.model import (
    create_baseline_model,
    get_model_info
)
from utils.evaluation import (
    evaluate_with_multiple_thresholds
)
from utils.profiling import (
    PerformanceProfiler,
    measure_time
)
from utils.visualization import (
    plot_performance_profile,
    plot_batch_size_comparison
)
from utils.architecture_optimization import (
    create_optimized_model
)

In [3]:
# Set device and analyze hardware capabilities
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Check tensor core support for mixed precision - crucial for FP16 acceleration
    gpu_compute = torch.cuda.get_device_properties(0).major
    tensor_core_support = gpu_compute >= 7  # Volta+ architecture
    print(f"Tensor Core Support: {tensor_core_support}")
else:
    print("WARNING: CUDA not available - hardware acceleration will be limited")

print("Default hardware acceleration environment ready!")

# Verify ONNX Runtime GPU support
print(f"\nONNX Runtime available providers: {ort.get_available_providers()}")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB
Tensor Core Support: True
Default hardware acceleration environment ready!

ONNX Runtime available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


> **Getting ready for acceleration**: The checks above highlight two critical facts for our mission:
> 1. Our reference hardware has tensor core support, which can dramatically speed up 16-bit floating-point (FP16) calculations; for other hardware deployments, like CPUs that lack this feature, we would need to rely on different techniques (such as 8-bit integer quantization (INT8)) to achieve similar acceleration.
> 2. ONNX Runtime providers are available for our primary targets: CUDAExecutionProvider for GPU and CPUExecutionProvider for CPU. This allows us to benchmark on both platforms. For a true mobile or edge deployment, we would need to use a specialized package like ONNX Runtime Mobile, which is built separately to keep the application lightweight.
> 
> Our task is to meet SLAs on our current device, which means we must **_benchmark against the GPU_** to see if we've met our goals.

## Step 2: Load test data and optimized model with configuration

The model is needed for deployment, and the optimization results for comparison.

Test data is needed for both conversion and final performance testing.

In [4]:
!nvidia-smi

Mon Mar 23 07:30:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:1E.0 Off |                    0 |
| N/A   25C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Define dataset loading parameters
img_size = 64
batch_size = 128

# Load test dataset for final evaluation
test_loader = load_pneumoniamnist(
    split="test", 
    download=True, 
    size=img_size,
    batch_size=batch_size,
    subset_size=None
)

# Get sample batch for profiling
sample_images, sample_labels = get_sample_batch(test_loader)
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)

print(f"Test data loaded: {sample_images.shape} batch for hardware acceleration profiling")

Using downloaded and verified file: /voc/work/.medmnist/pneumoniamnist_64.npz


Test data loaded: torch.Size([128, 3, 64, 64]) batch for hardware acceleration profiling


> **Batch size strategy**: Your batch size choice impacts memory usage, latency, and throughput. 
> 
> Consider: What batch size best applied for each deployment scenario? Don't forget to review the batch analysis plot from Notebook 2!

In [6]:
# Load optimized model and results from notebook 2

# TODO: Define the experiment name
experiment_name = "interpolation_removal_depthwise_sep_custom_exp5"
with open(f'../results/optimization_results_{experiment_name}.pkl', 'rb') as f:
    optimization_results = pickle.load(f)

print("Loaded optimization results from Notebook 2:")
print(f"   Model: {optimization_results['model_name']}")
print(f"   Clinical Performance: {optimization_results['clinical_performance']['optimized']['sensitivity']:.1%} sensitivity")
print(f"   Architecture Speedup: {optimization_results['performance_improvements']['latency_speedup']:.2f}x")
print(f"   Memory Reduction: {optimization_results['performance_improvements']['memory_reduction_percent']:.1f}%")

Loaded optimization results from Notebook 2:
   Model: ResNet-18 Optimized
   Clinical Performance: 99.0% sensitivity
   Architecture Speedup: 1.41x
   Memory Reduction: 78.3%


> **HINT: Finding your optimization results**
> 
> Your optimization results from Notebook 2 should be saved as:
> - Results file: `../results/optimization_results_{experiment_name}.pkl`
> - Model weights: `../results/optimized_model.pth`
> 
> The experiment name typically combines your optimization techniques, like:
> - `"interpolation-removal_depthwise-separable"`
> - `"channel-reduction_grouped-conv"`

In [7]:
class ResNet18(nn.Module):
    def __init__(self):
        super().__init__()        
        self.conv_block_0 = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1,  bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            )
        
        ###Residual Block 2 Reduction required in first Conv2D in first block component
        self.shortcut_3 = nn.Sequential(
            nn.Conv2d(64, 128, 1, padding=0, stride=2, bias=False),
            nn.BatchNorm2d(128)
            )
        
        self.residual_block_2_0 = nn.Sequential(
                nn.Conv2d(64, 160, 1, bias=False),
                nn.BatchNorm2d(160),
                nn.ReLU(inplace=True),
                nn.Conv2d(160, 160, 3, padding=1, stride=2, groups=160, bias=False),
                nn.BatchNorm2d(160),
                nn.ReLU(inplace=True),
                nn.Conv2d(160, 128, 1, bias=False),
                nn.BatchNorm2d(128)
                )
        self.residual_block_2_1 = nn.Sequential(
                nn.Conv2d(128, 160, 1, bias=False),
                nn.BatchNorm2d(160),
                nn.ReLU(inplace=True),
                nn.Conv2d(160, 160, 3, padding=1, groups=160, bias=False),
                nn.BatchNorm2d(160),
                nn.ReLU(inplace=True),
                nn.Conv2d(160, 128, 1, bias=False),
                nn.BatchNorm2d(128)
                )
        
        
        ###Residual Block 3 Reduction required in first Conv2D in first block component
        
        self.shortcut_5 = nn.Sequential(
            nn.Conv2d(128, 256, 1, padding=0, stride=2, bias=False),
            nn.BatchNorm2d(256)
            )
        self.residual_block_3_0 = nn.Sequential(
                nn.Conv2d(128, 320, 1, bias=False),
                nn.BatchNorm2d(320),
                nn.ReLU(inplace=True), 
                nn.Conv2d(320, 320, 3, padding=1, stride=2, groups=320, bias=False),
                nn.BatchNorm2d(320),
                nn.ReLU(inplace=True),
                nn.Conv2d(320, 256, 1, bias=False),
                nn.BatchNorm2d(256),
                )

        self.residual_block_3_1 = nn.Sequential(
                nn.Conv2d(256, 320, 1, bias=False),
                nn.BatchNorm2d(320),
                nn.ReLU(inplace=True), 
                nn.Conv2d(320, 320, 3, padding=1, groups=320, bias=False),
                nn.BatchNorm2d(320),
                nn.ReLU(inplace=True),
                nn.Conv2d(320, 256,1, bias=False),
                nn.BatchNorm2d(256)
                )
        ##Residual Block 4 Reduction required in first Conv2D in first block component
        
        self.shortcut_7 = nn.Sequential(
            nn.Conv2d(256, 512, 1, padding=0, stride=2, bias=False),
            nn.BatchNorm2d(512)
            )
        
        self.residual_block_4_0 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, stride=2, groups=256, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            )
        self.residual_block_4_1 = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1, groups=512, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            )
        self.aggregate = nn.AdaptiveAvgPool2d((1,1))
        
        self.classifier = nn.Sequential(
            nn.Dropout2d(0.1),
            nn.Conv2d(512, 2, 1),
            nn.Flatten()
        )
        self.input_size = 64
        
    def forward(self, x):

        x = self.conv_block_0(x)
        
        x_shortcut_3 = self.shortcut_3(x)
        x = self.residual_block_2_0(x)
        x = x + x_shortcut_3
        x = nn.ReLU(inplace=True)(x)
        x_shortcut_4 = x
        x = self.residual_block_2_1(x)
        x = x + x_shortcut_4
        x =  nn.ReLU(inplace=True)(x)
        
        x_shortcut_5 = self.shortcut_5(x)
        x = self.residual_block_3_0(x)
        x = x + x_shortcut_5
        x = nn.ReLU(inplace=True)(x)
        x_shortcut_6 = x
        x = self.residual_block_3_1(x)
        x = x + x_shortcut_6
        x = nn.ReLU(inplace=True)(x)
        x_shortcut_7 = self.shortcut_7(x)
        x = self.residual_block_4_0(x)
        x = x + x_shortcut_7
        x = nn.ReLU(inplace=True)(x)
        x_shortcut_8 = x
        x = self.residual_block_4_1(x)
        x = x + x_shortcut_8
        x = nn.ReLU(inplace=True)(x)
        x = self.aggregate(x)

        x = self.classifier(x)
        return x



In [8]:
# Get the optimization configuration
opt_config = optimization_results['optimization_config']
optimized_model = ResNet18()  

# TODO: Load the optimized model in the optimized_model variable
# HINT: This involves:
# > 1. Recreate the baseline model
# > 2. Applying the same architectural modifications using the saved optimization configuration
# > 3. Loading the trained weights
# See https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-model-for-inference for inspiration

# Add your code here
optimized_model.load_state_dict(torch.load('../results/optimized_model_best_dws_exp5_pk_mem.pth'))

<All keys matched successfully>

## Step 3: Convert model with hardware acceleration for production deployment

Convert the optimized model to [ONNX (Open Neural Network Exchange)](https://onnx.ai/) with optional hardware accelerations. 

**IMPORTANT**: You are tasked to implement both hardware optimizations even if you decide to disable them for the final export.

In [9]:
# TODO: Define your deployment configuration for the ONNX export.
# GOAL: Decide whether to use mixed precision (FP16) and/or dynamic batching for the final export.
# HINT: Setting use_fp16 to True can significantly improve performance on compatible GPUs (like the T4 with Tensor Cores)
# but may introduce a minor, often negligible, loss in precision. We'll validate the clinical impact later.

use_fp16 = True # Boolean; Set to True to enable mixed precision, False for standard FP32.
use_dynamic_batching = True # Boolean; Set to True to allow variable batch sizes, False for a fixed batch size.

In [10]:
sample_images.shape

torch.Size([128, 3, 64, 64])

In [11]:
# Convert PyTorch model to ONNX format (for cross-platform deployment)

def export_model_to_onnx(model: nn.Module, input_tensor: torch.Tensor, 
                        export_path: str, model_name: str = "pneumonia_detection", 
                        fp16_mode: bool = use_fp16, dynamic_batching: bool = use_dynamic_batching) -> str:
    """
    Export PyTorch model to ONNX format for production deployment.
    Apply hardware optimizations if selected.
    
    Args:
        model: PyTorch model to export
        input_tensor: Sample input tensor for shape inference
        export_path: Directory to save the ONNX model
        model_name: Name for the exported ONNX file
        fp16_mode: If True, exports the model in FP16 (mixed precision)
        dynamic_batching: If True, configures the model to accept variable batch sizes
        
    Returns:
        Path to exported ONNX model
    """
    # Define output path, and ensure it exists
    onnx_path = f"{export_path}/{model_name}.onnx"
    Path(export_path).mkdir(parents=True, exist_ok=True)
    
    # Convert PyTorch model to ONNX format for cross-platform deployment following the steps below
    # ONNX provides compatibility with TensorRT, OpenVINO, and other inference engines
    
    # 1. TODO: Set model to evaluation mode
    # Add your code here
    model.eval()

    # 2. TODO: Define the logic for fp16 mode
    # HINT: Think about what needs to be converted to half precision (input, model, or both?)
    # Add your code here
    if fp16_mode:
        model = model.half().to('cuda')
        input_tensor = input_tensor.half().to('cuda')
        print("model and input tensor converted to half precision")

    print(f"Exporting model to ONNX format...")
    print(f"   Input shape: {input_tensor.shape}")
    print(f"   Input dtype: {input_tensor.dtype}")
    print(f"   FP16 mode: {fp16_mode}")
    print(f"   Export path: {onnx_path}")
    
    dynamic_axes = None
    # 3. TODO: Define the logic for dynamic batching
    # HINT: Find the export argument in torch.onnx.export that supports setting dynamic axes
    # If you are not setting dynamic batching, how does onnx runtime choose the fixed batch size? Look at the input tensor in this case
    # Add your code here
    if dynamic_batching:
        dynamic_axes = {
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
            }
        print("All set for dynamic batching")

    # 4. Export to ONNX format with defined parameters
    torch.onnx.export(
        model,
        input_tensor,  # Input example
        onnx_path,
        export_params=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes=dynamic_axes,
        opset_version=16,  # Compatible with most inference engines
        do_constant_folding=True,  # Optimize constant operations
        verbose=False
    )
    
    print(f"ONNX export completed: {onnx_path}")

    # Verify ONNX model integrity - sanity check
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print("   ONNX model verification passed")
    except Exception as e:
        print(f"   WARNING: ONNX verification failed: {str(e)}")

    return onnx_path

# Export the mixed precision model to ONNX
onnx_model_path = export_model_to_onnx(
    model=optimized_model,
    input_tensor=sample_images,
    export_path="../results/onnx_models",
    model_name="udacimed_pneumonia_optimized_dws_exp5_pk_mem"
)

model and input tensor converted to half precision
Exporting model to ONNX format...
   Input shape: torch.Size([128, 3, 64, 64])
   Input dtype: torch.float16
   FP16 mode: True
   Export path: ../results/onnx_models/udacimed_pneumonia_optimized_dws_exp5_pk_mem.onnx
All set for dynamic batching
ONNX export completed: ../results/onnx_models/udacimed_pneumonia_optimized_dws_exp5_pk_mem.onnx
   ONNX model verification passed


## Step 4: Deploy with ONNX Runtime

With our model saved in the ONNX format, we can now load it into the [ONNX Runtime (ORT)](https://onnxruntime.ai/getting-started). 

ORT is a high-performance inference engine that can execute models on different hardware backends through its **Execution Providers (EPs)**. 

In [12]:
# This function creates an ONNX Runtime Inference Session.

# TODO: Choose whether the session should run on GPU or not
use_gpu = True # Boolean; Add your value here

def create_inference_session(model_path: str, use_gpu: bool = use_gpu) -> ort.InferenceSession:
    """
    Creates an ONNX Runtime inference session.

    Args:
        model_path: Path to the ONNX model file.
        use_gpu: If True, configures the session to use the CUDA Execution Provider.

    Returns:
        An ONNX Runtime InferenceSession object.
    """
    print(f"Creating ONNX Runtime session for {'GPU' if use_gpu else 'CPU'}...")
    
    # TODO: Define the execution providers
    # HINT: The `providers` argument takes a list of strings. For GPU, are you guaranteed that all operations can run on the CUDAExecutionProvider?
    # Reference: https://onnxruntime.ai/docs/performance/execution-providers/
    
    providers = []
    if use_gpu and torch.cuda.is_available():
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] # Add your code here
    else:
        providers = ['CPUExecutionProvider']
    
    # TODO: Create the ONNX Runtime InferenceSession
    # HINT: Instantiate an InferenceSession with the correct Execution Provider for the target hardware and any other desired parameters
    # Reference: https://onnxruntime.ai/docs/api/python/api_summary.html#inferencesession
    session = ort.InferenceSession(model_path, providers=providers) # Add your code here
    
    print(f"Session created with providers: {session.get_providers()}")
    return session

# Create the session for our exported ONNX model.
# We will run this on the GPU as it's our primary target device.
inference_session = create_inference_session(onnx_model_path)

Creating ONNX Runtime session for GPU...
Session created with providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


# Step 5: Benchmark model performance on all metrics

Now that we have a hardware-accelerated inference session, it's time to measure its performance. 

Unlike a server-based approach, we will perform direct, client-side benchmarking. This gives us precise measurements of the model's raw inference speed and resource consumption on our target hardware.

In [13]:
# Define a helper function to get input details and type

def get_input_details(session: ort.InferenceSession) -> Tuple[str, Tuple, np.dtype]:
    """
    Gets the input name, shape, and dtype for an ONNX Runtime session.
    """
    input_details = session.get_inputs()[0]
    input_name = input_details.name
    
    # TODO: Check if the model is FP16 to set the correct numpy dtype
    # HINT: Make sure the input type matches the type specified for the session input
    # Reference: https://onnxruntime.ai/docs/api/python/api_summary.html#onnxruntime.InferenceSession.get_inputs
    is_fp16 = [True for inp in session.get_inputs() if inp.type == 'tensor(float16)'][0] # Add your code here
    
    # Determine the correct numpy dtype
    input_dtype = np.float16 if is_fp16 else np.float32
    
    return input_name, input_details.shape, input_dtype

In [14]:
# This is the main benchmarking function.

def benchmark_performance(session: ort.InferenceSession, 
                          test_data: torch.Tensor,
                          batch_sizes: List[int],
                          num_runs: int = 50) -> Dict[str, Any]:
    """
    Benchmarks the performance of an ONNX Runtime session.

    Args:
        session: The ONNX Runtime inference session.
        test_data: A batch of test data for inference.
        batch_sizes: A list of batch sizes to test.
        num_runs: The number of inference runs to average for timing.

    Returns:
        A dictionary containing the performance results for each batch size.
    """
    results = {}
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    input_name, _, input_dtype = get_input_details(session)
    print(f"Benchmarking with input dtype: {input_dtype}")

    for batch_size in batch_sizes:
        print(f"--- Benchmarking Batch Size: {batch_size} ---")
        
        # Prepare batch data
        input_array = test_data[:batch_size].cpu().numpy().astype(input_dtype)
        
        # Warm-up runs to stabilize GPU clocks and cache
        for _ in range(10):
            session.run([output_name], {input_name: input_array})
            
        # Timed runs
        latencies = []
        
        # Perform the timed inference runs
        for _ in range(num_runs):
            start_time = time.perf_counter()
            session.run([output_name], {input_name: input_array})
            end_time = time.perf_counter()
            latencies.append((end_time - start_time) * 1000)  # Convert to ms
            
        # Measure peak GPU memory usage
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            # Run one more inference to capture memory usage after reset
            session.run([output_name], {input_name: input_array})
            peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
        else:
            peak_memory_mb = 0  # No GPU memory to measure on CPU

        # Calculate metrics
        avg_latency_ms = np.mean(latencies)
        throughput_sps = (batch_size / avg_latency_ms) * 1000  # Samples per second

        results[batch_size] = {
            'avg_latency_ms': avg_latency_ms,
            'throughput_sps': throughput_sps,
            'peak_memory_mb': peak_memory_mb
        }
        print(f"  Avg Latency: {avg_latency_ms:.3f} ms")
        print(f"  Throughput: {throughput_sps:,.2f} samples/sec")
        print(f"  Peak GPU Memory: {peak_memory_mb:.2f} MB")
        
    return results

# TODO: Define the batch size(s) you want to test.
# HINT: Powers of two are often optimal for GPU hardware, and 1 is useful for latency
batch_sizes_to_test = [1, 32, 64, 128]  # Add your values here

# Run the benchmark
benchmark_results = benchmark_performance(
    session=inference_session,
    test_data=sample_images,
    batch_sizes=batch_sizes_to_test
)

Benchmarking with input dtype: <class 'numpy.float16'>
--- Benchmarking Batch Size: 1 ---


  Avg Latency: 1.131 ms
  Throughput: 884.07 samples/sec
  Peak GPU Memory: 7.86 MB
--- Benchmarking Batch Size: 32 ---
  Avg Latency: 1.352 ms
  Throughput: 23,667.72 samples/sec
  Peak GPU Memory: 7.86 MB
--- Benchmarking Batch Size: 64 ---
  Avg Latency: 1.755 ms
  Throughput: 36,463.07 samples/sec
  Peak GPU Memory: 7.86 MB
--- Benchmarking Batch Size: 128 ---
  Avg Latency: 2.996 ms
  Throughput: 42,724.40 samples/sec
  Peak GPU Memory: 7.86 MB


## Step 6: Assess if production targets are met

Final evaluation against all production deployment requirements. Meeting all targets demonstrates successful optimization for UdaciMed's deployment requirements.

In [15]:
# Define production targets
# Note that we are skipping FLOP analysis here because not directly impacted by hardware acceleration
PRODUCTION_TARGETS = {
    'memory': 100,               # MB - Achievable with mixed precision
    'throughput': 2000,          # samples/sec - Target for multi-tenant deployment
    'latency': 3,                # ms - Individual inference time for real-time scenarios
    'sensitivity': 98,           # % - Clinical safety requirement (non-negotiable)
}

In [16]:
# STEP 1: Extract the best batch configuration from the benchmark results

# Initialize variables to hold the best results found.
latency_for_target = float('inf')
max_throughput = 0
best_throughput_bs = None
memory_at_max_throughput = 0

# Check if the real-time latency scenario (batch size 1) was tested.
if 1 in benchmark_results:
    latency_for_target = benchmark_results[1]['avg_latency_ms']
else:
    print("WARNING: Batch size 1 not found in results. Real-time latency target cannot be evaluated.")

# Find the batch size that yielded the highest throughput.
if benchmark_results:
    best_throughput_bs = max(benchmark_results, key=lambda bs: benchmark_results[bs]['throughput_sps'])
    max_throughput = benchmark_results[best_throughput_bs]['throughput_sps']
    memory_at_max_throughput = benchmark_results[best_throughput_bs]['peak_memory_mb']

# Get model file size as another memory metric
model_file_size_mb = Path(onnx_model_path).stat().st_size / (1024 * 1024)

print("\n--- Performance Analysis ---")
print(f"Real-time Latency (BS=1): {f'{latency_for_target:.3f} ms' if latency_for_target != float('inf') else 'Not Tested'}")
if best_throughput_bs is not None:
    print(f"Max Throughput: {max_throughput:,.2f} samples/sec (at Batch Size={best_throughput_bs})")
    print(f"Peak GPU memory at max throughput: {memory_at_max_throughput:.2f} MB")
print(f"Model file size: {model_file_size_mb:.2f} MB")


--- Performance Analysis ---
Real-time Latency (BS=1): 1.131 ms
Max Throughput: 42,724.40 samples/sec (at Batch Size=128)
Peak GPU memory at max throughput: 7.86 MB
Model file size: 1.82 MB


In [17]:
# STEP 2: Define a function to validate the clinical performance using the ONNX session.

def validate_clinical_performance(session: ort.InferenceSession, 
                                  test_loader, 
                                  threshold: float = 0.5) -> Dict[str, Any]:
    """
    Validates clinical performance (sensitivity) using the ONNX Runtime session.
    """
    print("\nValidating clinical performance on test data...")
    input_name, _, input_dtype = get_input_details(session)
    output_name = session.get_outputs()[0].name

    all_predictions = []
    all_labels = []

    for batch_inputs, batch_labels in test_loader:
        # Prepare input
        input_array = batch_inputs.cpu().numpy().astype(input_dtype)
        
        # Run inference
        results = session.run([output_name], {input_name: input_array})
        logits = torch.from_numpy(results[0])
        
        # Process output
        probabilities = torch.softmax(logits, dim=1)[:, 1] # Probability of class 1 (pneumonia)
        all_predictions.extend(probabilities.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

    # Calculate metrics
    predictions = np.array(all_predictions)
    labels = np.array(all_labels).flatten()
    pred_classes = (predictions > threshold).astype(int)
    
    tp = np.sum((pred_classes == 1) & (labels == 1))
    fn = np.sum((pred_classes == 0) & (labels == 1))
    
    sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"Clinical validation completed on {len(labels)} samples.")
    print(f"  Calculated Sensitivity: {sensitivity:.2f}% (at threshold={threshold})")
    
    return {'sensitivity': sensitivity}


# TODO: Choose a clinical threshold for classification.
# GOAL: Set a decision threshold for classifying a case as pneumonia.
# HINT: This value is often determined through clinical studies. A higher threshold
# might reduce false positives but could lower sensitivity. We need to ensure we
# still meet the sensitivity target with the chosen value.
clinical_threshold = 0.7 # Float; Add your value here 

clinical_results = validate_clinical_performance(
    session=inference_session,
    test_loader=test_loader,
    threshold=clinical_threshold
)



Validating clinical performance on test data...


Clinical validation completed on 624 samples.
  Calculated Sensitivity: 98.97% (at threshold=0.7)


In [18]:
# TODO: Manually set the FLOPS target % reduction met given your results from Notebook 2
flops_target_reduction = 80
flops_achieved_reduction = 98.97 # Float (%); Add your value here
flp_ok = True # Boolean; Add your value here

# Check if targets are met
mem_ok = model_file_size_mb < PRODUCTION_TARGETS['memory']
lat_ok = latency_for_target < PRODUCTION_TARGETS['latency']
thr_ok = max_throughput > PRODUCTION_TARGETS['throughput']
sen_ok = clinical_results['sensitivity'] > PRODUCTION_TARGETS['sensitivity']
all_ok = all([mem_ok, lat_ok, thr_ok, sen_ok, flp_ok])

print(f"| Metric          | Target                    | Achieved                  | Status  |")
print(f"|-----------------|---------------------------|---------------------------|---------|")
print(f"| Memory          | < {PRODUCTION_TARGETS['memory']} MB                  | {model_file_size_mb:.2f} MB                   | {'✔️ Met' if mem_ok else '✖️ Missed'}  |")
print(f"| Latency         | < {PRODUCTION_TARGETS['latency']} ms                    | {latency_for_target:.3f} ms                  | {'✔️ Met' if lat_ok else '✖️ Missed'}  |")
print(f"| Throughput      | > {PRODUCTION_TARGETS['throughput']:,} samples/sec       | {max_throughput:,.2f} samples/sec     | {'✔️ Met' if thr_ok else '✖️ Missed'}  |")
print(f"| FLOP Reduction  | > {flops_target_reduction}%                     | {flops_achieved_reduction:.1f}%                     | {'✔️ Met' if flp_ok else '✖️ Missed'}  |")
print(f"| Sensitivity     | > {PRODUCTION_TARGETS['sensitivity']}%                     | {clinical_results['sensitivity']:.2f}%                    | {'✔️ Met' if sen_ok else '✖️ Missed'}  |")
print(f"\nOverall Result: {'CONGRATS: All production targets met!' if all_ok else 'WARNING: Some targets were not met. Further optimization may be needed.'}")
print(f"\nNOTE: This analysis does not consider FLOPs which can are not improved through hardware acceleration; please check your results on this metric from notebook 2")

| Metric          | Target                    | Achieved                  | Status  |
|-----------------|---------------------------|---------------------------|---------|
| Memory          | < 100 MB                  | 1.82 MB                   | ✔️ Met  |
| Latency         | < 3 ms                    | 1.131 ms                  | ✔️ Met  |
| Throughput      | > 2,000 samples/sec       | 42,724.40 samples/sec     | ✔️ Met  |
| FLOP Reduction  | > 80%                     | 99.0%                     | ✔️ Met  |
| Sensitivity     | > 98%                     | 98.97%                    | ✔️ Met  |

Overall Result: CONGRATS: All production targets met!

NOTE: This analysis does not consider FLOPs which can are not improved through hardware acceleration; please check your results on this metric from notebook 2


---

## Step 7: Cross-platform deployment analysis

We have successfully optimized our model to meet _UdaciMed's Universal Performance Standard_ on our standardized target device. 

With ONNX, we can easily deploy this optimized model across UdaciMed's diverse hardware fleet just by [changing the Execution Providers](https://onnxruntime.ai/docs/execution-providers/):

| Deployment Target	| Recommended Technology |	Primary Goal	 |	Key Trade-Off | 
| :--- | :--- | :--- | :--- |
| GPU Server (Cloud/On-Prem) |		ONNX Runtime + TensorRT		 |Max Throughput 	 |	Highest performance vs. more complex setup. | 
| CPU Workstation (Hospital) |		ONNX Runtime + OpenVINO		 |Low Latency  |		Excellent CPU speed vs. being tied to Intel hardware. | 
| Mobile/Edge Device (Clinic) |		ONNX Runtime Mobile		 | Small Footprint  |		Maximum portability vs. reduced model precision (quantization). | 

But **what if we need to squeeze out every last drop of performance from each deployment target?** To do this, let's consider moving beyond the portable ONNX format and use specialized, hardware-specific frameworks.

### **Step 7.1: Optimization strategy for specialized GPU server deployment**

We've established a strong performance baseline using the standard ONNX Runtime with its CUDA Execution Provider (EP). 

Now, let's explore more advanced options to see if we can unlock even greater performance or add production-grade features for our high-demand GPU deployments.

#### TODO: Analyze GPU Deployment Options

For a production environment, we need to decide not just if we use a GPU, but _how we use it_.

_<\<Complete the table below by filling in missing performance expectations\>>_

| Approach | How it Works | Key Performance Contributor | Complexity/Overhead | UdaciMed Suitability |
| :--- | :--- | :--- | :--- | :--- |
| **ONNX Runtime with CUDA Execution Provider** | _(Our Baseline)_ Executes the ONNX graph directly on the GPU using CUDA libraries. | Good (fast, direct GPU access) | Low (simple library integration) | Excellent for direct application integration. |
| **ONNX Runtime with TensorRT Execution Provider** | converts supported parts of the ONNX graph into an optimized TensorRT engine and then executes that engine on GPU |large batch size, precision mode,tensorrt partitions, layer fusion(Conv + BatchNorm + ReLU)  | fragmented graphs if unspported nodes in onnx graph, tensorrt does not works good with dynamic batch shapes, tricky(version compatibility complexity) |difficult to implement owing to version comptibility issues between onnx and tensorrt
| **Triton Inference Server with TensorRT backend** |Triton uses TensorRT as an inference engine backend while adding production-grade scheduling, batching, concurrency, and serving infrastructure around it.  |triton server helps with dynamic batching resulting in more efficient tensorrt kernel launches, precision mode  |serialization overhead along with CPU-GPU transfers |good integration of triton with tensorrt engine

_<<Briefly answer the questions below based on UdaciMed's hospital deployment requirements>>_

**1. What is the main business risk of choosing the TensorRT path over the CUDA EP baseline?**

Compatibility and version conflict issues between onnx and tensorrt plus tensorrt does not works good with dynamic batch shapes.

**2. Why might a small clinic with a single on-premise GPU workstation not want the complexity of Triton, even if it offers advanced features?**

Specialized implementation, costly to implement and support. Triton is designed for enterprise level deployment perhaps an overkill for single GPU workstation

#### TODO: Make your strategic choice

Based on your analysis, choose the best GPU server deployment approach for UdaciMed's long-term goal of a multi-tenant service.

For multitenant service, using onnx with GPU execution provider with CPU Execution provider as fall back strategy provides a good deployment choice in long run as you might want something to run reliably on a variety of hardware

**My recommendation for UdaciMed's GPU server deployment:** 

I will go with ONNX with cuda EP as it is more easy to implement, provides good performance and is compatible across variety of hardware. Triton with TensorRT also provides good integration on cuda devices.

#### TODO: Fix this Triton Inference Server configuration 

Explain how to extend the following Triton configuration to introduce mixed-precision and dynamic batching.

```config.pbtxt

name: "udacimed_pneumonia_prod"
platform: "onnxruntime_onnx"
max_batch_size: 64

dynamic_batching {
  preferred_batch_size: [ 4, 8, 16, 32, 64 ]  # optional, list of preferred batch sizes
  max_queue_delay_microseconds: 10000        # wait time (10 ms) for batching
}

input [
  {
    name: "input"
    data_type: TYPE_FP16  
    dims: [ 3, 64, 64 ]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP16 
    dims: [ 2 ]
  }
]
```

<<Review the Triton documentation and explain how to add the requested hardware accelerations in 1-2 sentences.>>
1) Change the data type to FP16
2) Add dynamic batching block with different batch sizes and max wait time for batching operation

### **Step 7.2: Optimization strategy for specialized CPU deployment**

Deploying on CPUs is critical for UdaciMed's success, as most hospitals and clinics rely on standard workstations without dedicated GPUs. Let's analyze CPU options for UdaciMed's hospital deployment!

> **Numerical precision opportunities with GPU and CPU**: CPUs don't benefit from FP16 (most CPUs only emulate FP16). But CPUs supports another type of numerical optimization, remember?

#### TODO: Analyze CPU deployment options

While our ONNX model can run on any CPU, using specialized execution providers can unlock significant performance gains, especially on Intel hardware.

_<\<Complete the table below by filling in missing performance expectations\>>_

| Approach | How it Works | Conversion Path | Memory Footprint | Performance | UdaciMed Suitability |
|----------|--------------|-----------------|------------------|-------------| ---------------------| 
| **PyTorch on CPU** | The original, un-optimized model running directly on the CPU.| Direct (no conversion) | High (includes Python interpreter overhead)| Baseline (slowest) | A good reference point, but not for production. |
| **ONNX Runtime with Default CPU** | by utilizing its dedicated CPUExecutionProvider |once pytorch model is converted into onnx model, onnx automatically uses it based on configured execution providers  | 2x to 4x the model size  |faster than pytorch on cpu through graph optimization and 8bit quanitization  | can be used on CPU based machines suitable to run production workload|
| **ONNX Runtime with OpenVINO** |by utilizing the OpenVINO Execution Provider (EP), which accelerates inference of ONNX models on Intel hardware (CPUs, GPUs, VPUs)  |pytorch model is converted into onnc format which is then dynamically consumed and optimized by OpenVINO at runtime, or explicitly converted to OpenVINO Intermediate Representation (IR) for maximum performance.  | depends on model size, batch size, precision, and OpenVINO optimizations  | delivers significant performance improvements, often up to 10× faster, for AI models running on Intel hardware (CPU, iGPU, VPU | good for intel servers suitable for production deployment |
| **OpenVINO** |optimizes and converts pre-trained AI models into an Intermediate Representation (IR) format, which the Inference Engine then executes across Intel hardware for maximum performance | load trained models from pytorch/onnx into the OpenVINO Intermediate Representation (IR) format  | OpenVINO is designed to be lightweight, with a runtime footprint often just a few hundred megabytes for dependencies. Memory usage depends on the device (CPU vs. GPU) and model size, with techniques like INT8 quantization, sparse weight decompression, and model caching used to minimize RAM consumption. |high-performance AI inference, specifically optimized for Intel hardware (CPUs, GPUs, NPUs)| good for intel servers suitable for production deployment |
| **OpenVINO Backend for Triton** |acts as a wrapper that allows Triton to run AI models on Intel CPUs, integrated GPUs, and other Intel-made AI accelerators |Original Model (ONNX/TensorFlow/PyTorch) → OpenVINO Model Optimizer → IR (.xml + .bin) → Triton Model Repository → Triton OpenVINO Backend → Optimized Inference on Intel Hardware  |add triton overhead to normal openvino  | high performance inference on Intel CPU and accelerator hardware by leveraging hardware-specific optimizations. | good for intel servers suitable for production deployment |

_<\<Briefly answer the questions below based on UdaciMed's hospital deployment requirements>>_

**1. What is the key advantage of converting the model to "Native OpenVINO IR" over simply using the ONNX + OpenVINO EP, and when would it be worth the extra effort?**
IR format allows full OpenVINO graph optimizations, precision conversions, and hardware-specific kernel generation, which can significantly improve inference speed, memory efficiency, and deployment stability.

**2. Triton Server has the "Highest" memory overhead. When would it ever make sense to use it for a CPU-based deployment?**
You use Triton on CPU when you want production-ready inference with concurrency, batching, monitoring, and multi-model support, even if memory usage is higher. Its overhead is the price for robustness and scalability.

**3. No matter which of the five options is chosen, what is the single most important metric to re-validate to ensure clinical safety?**
sensitivity along with f1 score

#### TODO: Make your strategic choice

Based on your analysis, choose the best CPU deployment approach for UdaciMed's typical hospital workstation client.

OpenVINO for high performance and tight integration with intel based hardware

#### TODO: Define an optimal CPU deployment configuration in OpenVINO

Imagine you are testing out CPU deployment with OpenVINO for UdaciMed, and set up the OpenVINO configuration to balance performance, memory, and clinical safety.

_<\<Complete the OpenVINO configuration below>>_

```yaml
# openvino_hospital_config.yaml
# UdaciMed Hospital Workstation Deployment Configuration

model_optimization:
  input_model: "udacimed_pneumonia_optimized.onnx"
  target_device: "CPU"
  
  # Choose precision strategy
  precision: "FP32" # TODO - Options: "FP32" (safe), "FP16", or "INT8" (faster, smaller, but clinical risk)
  #justification: FP32 ensures maximal clinical safety and no loss of diagnostic accuracy.
  
  # Set optimization priority  
  optimization_level: "ACCURACY" # TODO - Options: "ACCURACY" (safe) or "PERFORMANCE" (faster)
  #justification:  Prioritize accuracy over raw performance for patient safety
  
  # Configure quantization (if using INT8)
  quantization:
    enabled: false # TODO: true/false 
    #justification: INT8 is not used to avoid clinical risk due to reduced numerical precision.
    calibration_dataset_size: 0  # TODO - Number of samples for INT8 calibration (if enabled)
    #justification -> not required as quantization is disabled

deployment_config:
  # Configure CPU utilization for hospital workstations
  cpu_threads: 4 # TODO - Options: 1, 2, 4, 8 (consider multi-tenancy impact)
  #justification: Moderate threading balances performance and avoids saturating shared workstation CPU.

  
  # Set memory allocation for multi-tenant deployment
  memory_pool_mb: 1024 # TODO - Memory budget per model instance
  #justification: 1 GB per model instance ensures stable performance without starving other hospital software.

  # Choose batching strategy
  max_batch_size: 1 # TODO - 1 (single patient) or higher (if implementing manual batching)
  #justification: Single patient inference to minimize latency and avoid clinical batching errors.
  
  # Configure for hospital network environment
  inference_timeout_ms:500   # TODO: Maximum inference time before timeout
  #justification: 500 ms timeout prevents delays in real-time diagnostic workflows.

clinical_validation:
  # Define validation requirements after CPU deployment
  sensitivity_threshold: 0.985 # TODO: Minimum acceptable sensitivity (should be >98%)
  # justification: Maintains greater than 98% sensitivity to ensure clinical safety.
  validation_dataset_size: 600 # TODO: Number of samples for clinical re-validation
  #justification: Large enough dataset to verify model accuracy post-deployment.
  comparison_baseline: "GPU_Triton_deployment"  # Compare against your GPU results
```

_<\<Justify each configuration choice with one sentence each>>_

### **Step 7.3: Optimization strategy for mobile and edge deployment**

UdaciMed's vision extends beyond hospital workstations to portable devices and mobile health applications. This enables pneumonia detection in rural clinics, emergency response, and preventive screening programs where traditional infrastructure is limited.

> **Mobile and edge requirements**: These deployments require lightweight runtimes, offline capability, extended battery life, and often benefit from platform-specific optimizations. However, conversion complexity and clinical validation requirements vary significantly across approaches.

#### TODO: Analyze mobile deployment options

For mobile, the choice between a cross-platform solution and a native, OS-specific framework is the most critical decision, with significant long-term consequences for development and user experience.

Here, the primary constraints are not raw speed, but model size, power consumption, and offline capability. We need a model that is small, efficient, and fully self-contained.

_<\<Complete the table below by filling in missing performance expectations\>>_

| Platform | How it Works | Key Strength | Main Trade-Off | UdaciMed Suitability |
|----------|----------------|------------|---------------|-------------------|
| **ONNX Runtime Mobile** | A cross-platform engine runs a single ONNX file on iOS & Android. | Portability & simplicity | Not the most optimized performance	 | Best for a fast, low-budget launch to reach all users. |
| **ExecuTorch** | using an ahead-of-time (AOT) compilation workflow to convert a standard PyTorch model into an optimized, standalone binary that can be efficiently run on resource-constrained edge devices  | ability to provide a unified, PyTorch-native, and high-performance workflow for deploying AI models directly to edge devices (mobile, wearables, and microcontrollers) without needing to convert to third-party formats like TFLite or ONNX. |less flexibility for dynamic models and potential debugging complexity| good fro mobile deployments |
| **LiteRT** |mini optimized engine that takes a model, cleans it up, fuses operations, and runs it efficiently on CPU/GPU with low memory and latency, especially for edge devices.  | high-performance, low-latency inference with minimal memory footprint, especially on edge and resource-constrained devices. | lack of flexibility and ease of debugging  | avoid |
| **Core ML (iOS)** |It allows developers to run trained ML models directly on Apple devices  |highly optimized on-device inference, combining speed, privacy, and energy efficiency. | platform locked | can be used on apple devices |

_<\<Answer the questions below based on UdaciMed's mobile and edge deployment strategy>>_

**1. What is the key trade-off between ONNX Runtime Mobile's "simplicity" and LiteRT's "smallest size & fastest speed"?**

onnx runtime mobile is slow and has high memory footprimt. LiteRT is less flexible when compared to onnx

**2. Which frameworks are best suited for a fully offline-capable app for use in rural clinics with no internet, and why?**

ExecuTorch - It provides excellent support for dynamic, complex, state-of-the-art models and is optimized for ARM CPUs using QNNPACK/XNNPACK

 Apple Core ML -  It is the most efficient framework for Apple hardware, directly utilizing the Apple Neural Engine, GPU, and CPU. It offers seamless integration and high performance.

ONNX Runtime: It boasts exceptional portability, running models on both Android and iOS with a focus on performance. Its ability to support models from PyTorch, TensorFlow, and Scikit-learn via the ONNX standard makes it highly flexible.

TFLite: It is explicitly designed for mobile/IoT devices, featuring a small footprint (~1MB) and high optimization for CPU, GPU, and Android NNAPI (neural network API) accelerators.

**3. For a battery-powered portable device, which frameworks would likely offer the best power efficiency, and what is the trade-off?**

LiteRT becasue it has extremely minimal runtime and memory footprint. Trade Off is its less flexibility.
Core ML for IOS becasue it is optimized for apple devices. it is platform locked. can be used only on apple devices

#### TODO: Make your strategic choice

Based on your analysis, choose the best mobile deployment approach for UdaciMed's initial launch.
TFLite.

**My recommendation for UdaciMed's mobile and edge deployment strategy:**

TFLite: Extremely minimal runtime and memory footprint resulting in lesser power consumption.

-----

## **Congratulations!**

You have successfully implemented a complete hardware-accelerated deployment pipeline! Let's recap the decisions you have made and results you have achieved while transforming an optimized model into a production-ready healthcare solution.

### **TODO: Production deployment scorecard**

**Final GPU deployment performance vs UdaciMed targets:**

_<\<Complete final scorecard based on your benchmarking results:>>_

| Metric | Target | Achieved | Status |
|--------|--------|----------|--------|
| **Memory Usage** | <100MB | yes| green|
| **Throughput** | >2,000 samples/sec |yes |green |
| **Latency** | <3ms | yes | green|
| **FLOP Reduction** | <0.4 GFLOPs per sample (80%) |yes |green|
| **Clinical Safety** | >98% sensitivity |yes |green|

_<\<Give yourself a final production score given the number of targets met>>_

**Overall production score: 5/5 targets met!**

### **TODO: Strategic deployment insights**

_<\<Reflect on the key decisions you made, and why>>_

#### Mixed Precision Strategy
**Your FP16/FP32 choice:** FP16

**Why you made this decision:**
Faster execution. Low memory footprint

#### Backend Selection
**Your ONNX execution provider choice:**  CudaEP backed by CPU EP

**Why this backend aligned with UdaciMed's requirements:**
wide variety of hardware plaforms
#### Batching Configuration
**Your dynamic batching setup**: 32  # _(preferred batch sizes, queue delay, etc.)_

**How this supports diverse clinical deployments:** 
small enough for speed and efficiency and supports diverse platforms 

### Optimization Philosophy
**Meeting targets vs maximizing metrics:**

Tradeoff is the real game. Model may have low latency but the peak memory consumption may be high. Then you work on improving peak memory without impacting latency and vice versa till you reach the desired sweat spot

---

**You have completed the full journey from architectural optimization to production-ready deployment, demonstrating the technical skills and strategic thinking essential for deploying AI in healthcare. Your UdaciMed pneumonia detection system is now ready to serve hospitals worldwide while maintaining the clinical safety standards that save lives.**